# 03 Búsqueda vectorial con TF-IDF

## Objetivo

- Calcular e interpretar TF, DF, IDF y TF-IDF.
- Implementar TF-IDF mediante operaciones explícitas y verificarlo con `TfidfVectorizer`.
- Calcular similitud coseno mediante el producto punto.
- Implementar BM25 y comprender sus componentes.
- Construir y comparar rankings para una misma consulta.

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
from collections import Counter
from pathlib import Path
import json
import random
import re
import unicodedata
import pandas as pd

TOKEN_RE = re.compile(r"[a-z]+(?:['\-][a-z]+)*")

def clean_special(text: str) -> str:
    # Normaliza comillas, guiones largos y quita acentos
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("’", "'").replace("‘", "'")
    text = text.replace("“", '"').replace("”", '"')
    text = re.sub(r"[—–_]+", " ", text)
    text = unicodedata.normalize("NFD", text)
    return "".join(c for c in text if unicodedata.category(c) != "Mn")

def preprocess(text: str) -> list[str]:
    # Minúsculas, limpieza y tokenización
    text = text.lower()
    text = clean_special(text)
    return TOKEN_RE.findall(text)

In [ ]:
CORPUS_DIR = Path("txt-files") / "epub"
MAX_DOCS = 1000
SEMILLA = 42
MIN_TOKENS = 100

# Marcadores que usa Gutenberg para delimitar el contenido real del libro
INICIO_RE = re.compile(r"\*\*\* ?START OF (?:THE|THIS) PROJECT GUTENBERG EBOOK[^\n]*\n", re.IGNORECASE)
FIN_RE = re.compile(r"\*\*\* ?END OF (?:THE|THIS) PROJECT GUTENBERG EBOOK", re.IGNORECASE)

def leer_texto(ruta: Path) -> str:
    # Lee el archivo según su codificación (-8.txt es latin-1)
    datos = ruta.read_bytes()
    cod = "latin-1" if ruta.name.lower().endswith("-8.txt") else "utf-8"
    return datos.decode(cod, errors="replace")

def extraer_titulo(texto: str, libro_id: int) -> str:
    # Busca la línea "Title: ..." del encabezado de Gutenberg
    m = re.search(r"^\s*Title:\s*(.+)", texto[:5000], re.MULTILINE)
    return m.group(1).strip() if m else f"Documento {libro_id}"

def quitar_encabezado_pie(texto: str) -> tuple[str, bool]:
    # Devuelve el cuerpo del libro y si se encontraron ambos marcadores
    inicio = INICIO_RE.search(texto)
    fin = FIN_RE.search(texto)
    if not inicio or not fin or fin.start() <= inicio.end():
        return texto, False
    return texto[inicio.end():fin.start()], True

def cargar_corpus(documentos_validos, max_docs=MAX_DOCS, semilla=SEMILLA, min_tokens=MIN_TOKENS):
    # Elige documentos al azar
    ids = sorted(documentos_validos)
    random.Random(semilla).shuffle(ids)
    ids = sorted(ids[:max_docs])

    textos, titulos, corpus = {}, {}, {}
    sin_marcadores, excluidos = [], []
    for libro_id in ids:
        completo = leer_texto(documentos_validos[libro_id])
        cuerpo, ok = quitar_encabezado_pie(completo)
        tokens = preprocess(cuerpo)

        if len(tokens) < min_tokens:         # Excluye si tiene muy pocos tokens
            excluidos.append(libro_id)
            continue
        if not ok:
            sin_marcadores.append(libro_id)

        titulos[libro_id] = extraer_titulo(completo, libro_id)
        textos[libro_id] = cuerpo
        corpus[libro_id] = tokens
    return textos, titulos, corpus, sin_marcadores, excluidos

# Carga del índice de documentos válidos
with open("documentos_validos.json", encoding="utf-8") as f:
    documentos_validos = {int(k): CORPUS_DIR / v for k, v in json.load(f).items()}

textos, titulos, corpus, sin_marcadores, excluidos = cargar_corpus(documentos_validos)
ids_docs = list(corpus)   # orden fijo de documentos para todo el cuaderno
print(f"{len(corpus)} documentos cargados | sin marcadores: {len(sin_marcadores)} | "f"excluidos (menos de {MIN_TOKENS} tokens): {len(excluidos)} {excluidos}")

989 documentos cargados | sin marcadores: 0 | excluidos (menos de 100 tokens): 11 [4577, 7337, 23816, 23878, 23950, 23951, 24225, 25229, 26904, 31978, 54784]


In [3]:
def tabla_corpus(corpus, titulos):
    # Tabla: ID | Título | Número de términos únicos
    filas = [(i, titulos[i], len(set(corpus[i]))) for i in corpus]
    return pd.DataFrame(filas, columns=["ID", "Título", "Términos únicos"])

def resumen_corpus(corpus):
    # Número de documentos, vocabulario y longitudes (en tokens)
    vocabulario = set()
    for tokens in corpus.values():
        vocabulario.update(tokens)
    longitudes = [len(t) for t in corpus.values()]
    print(f"Número de documentos : {len(corpus)}")
    print(f"Tamaño del vocabulario: {len(vocabulario):,}")
    print(f"Longitud mínima       : {min(longitudes):,}")
    print(f"Longitud media        : {sum(longitudes) / len(longitudes):,.1f}")
    print(f"Longitud máxima       : {max(longitudes):,}")

df_corpus = tabla_corpus(corpus, titulos)
display(df_corpus)
resumen_corpus(corpus)

,ID,Título,Términos únicos
0,10,The King James Version of the Bible,12780
1,329,Island Nights' Entertainments,4539
2,369,The outlaw of Torn,6425
3,391,The Song of Roland,4146
4,434,The Circular Staircase,6020
...,...,...,...
984,79627,The confessions of a dancing girl,4435
985,79647,About Zionism,2332
986,79667,Ganymede House,552
987,79790,My life and loves in Greenwich Village,9430


Número de documentos : 989
Tamaño del vocabulario: 1,171,922
Longitud mínima       : 107
Longitud media        : 63,308.1
Longitud máxima       : 4,634,329


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [ ]:
import math
from collections import Counter

    # Funcion calculo TF
def calcular_tf(corpus):
    tf = {}
    for doc_id, tokens in corpus.items():
        tf[doc_id] = Counter(tokens)
    return tf

def calcular_df(tf):
    # Funcion calculo DF
    df = Counter()
    for conteos in tf.values():
        for termino in conteos:      # las claves son únicas: cuenta 1 vez por documento
            df[termino] += 1
    return df

def calcular_idf(df, N):
    # Funcion calculo IDF: ln(N / df(t)) + 1 -> {término: idf}
    idf = {}
    for termino, d in df.items():
        idf[termino] = math.log(N / d) + 1
    return idf

def calcular_tfidf(tf, idf):
    # Funcion calculo TF-IDF: w(t,d) = TF(t,d) * IDF(t) -> {doc_id: {término: peso}}
    tfidf = {}
    for doc_id, conteos in tf.items():
        tfidf[doc_id] = {}
        for termino, frecuencia in conteos.items():
            tfidf[doc_id][termino] = frecuencia * idf[termino]
    return tfidf

### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [ ]:
import numpy as np

    # Vocabulario ordenado alfabéticamente
def construir_vocabulario(corpus):
    vocabulario = set()
    for tokens in corpus.values():
        vocabulario.update(tokens)
    return sorted(vocabulario)

vocab = construir_vocabulario(corpus)
indice_vocab = {t: j for j, t in enumerate(vocab)}
print(f"Vocabulario: {len(vocab):,} términos | primeros: {vocab[:5]}")

Vocabulario: 1,171,922 términos | primeros: ['a', "a'ali", "a'ana", "a'angee", "a'beckett"]


In [ ]:
N = len(corpus)

tf = calcular_tf(corpus)       
df = calcular_df(tf)           
idf = calcular_idf(df, N)       

idf_vec = np.array([idf[t] for t in vocab])
print(f"N = {N} | ejemplo: df['the'] = {df['the']}, idf['the'] = {idf['the']:.4f}")

In [ ]:
def matriz_tf(tf, ids_docs, indice_vocab):
    # Matriz TF
    M = np.zeros((len(ids_docs), len(indice_vocab)))
    for i, doc_id in enumerate(ids_docs):
        for termino, frecuencia in tf[doc_id].items():
            M[i, indice_vocab[termino]] = frecuencia
    return M

def matriz_tfidf(M_tf, idf_vec):
    # TF-IDF
    return M_tf * idf_vec

M_tf = matriz_tf(tf, ids_docs, indice_vocab)
X_tfidf = matriz_tfidf(M_tf, idf_vec)    
print("Forma de la matriz TF-IDF (documentos x términos):", X_tfidf.shape)

Forma de la matriz TF-IDF (documentos x términos): (989, 1171922)


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Uso libreria TfidfVectorizer
vectorizador = TfidfVectorizer(
    lowercase=False,
    token_pattern=r"\S+",
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
)

docs_texto = [" ".join(corpus[i]) for i in ids_docs]
X_sk = vectorizador.fit_transform(docs_texto).toarray()
print("Forma matriz scikit-learn:", X_sk.shape)

Forma matriz scikit-learn: (989, 1171922)


In [10]:
def comparar_matrices(X_propia, X_sklearn, vocab_propio, vocab_sklearn):
    # Revisa forma, orden de columnas y máxima diferencia absoluta
    print("Misma forma          :", X_propia.shape == X_sklearn.shape)
    print("Mismo orden de vocab :", list(vocab_propio) == list(vocab_sklearn))
    diferencia = np.abs(X_propia - X_sklearn).max()
    print(f"Máxima diferencia absoluta: {diferencia:.2e}")
    return diferencia

vocab_sk = vectorizador.get_feature_names_out()
dif_max = comparar_matrices(X_tfidf, X_sk, vocab, vocab_sk)

TOLERANCIA = 1e-9
assert dif_max < TOLERANCIA, "Las matrices no coinciden: revisar preprocesamiento, columnas, IDF o normalización"
print("Las matrices coinciden dentro de la tolerancia")

Misma forma          : True
Mismo orden de vocab : True
Máxima diferencia absoluta: 0.00e+00
Las matrices coinciden dentro de la tolerancia


In [11]:
del M_tf, X_sk

### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [26]:
consultas = {
    "clave": {   # palabras clave
        "Q1": ("detective murder mystery clue police",
               "Novelas de misterio o crimen donde un detective investiga un asesinato."),
        "Q2": ("fairy tale princess king castle dragon",
               "Cuentos de hadas o historias infantiles con princesas, reyes y dragones."),
        "Q3": ("ancient rome emperor empire senate",
               "Libros de historia sobre Roma antigua, sus emperadores y su imperio."),
        "Q4": ("recipe cook bake flour sugar",
               "Libros de cocina o recetas para preparar pan, pasteles y otros platos."),
    },
    "frase": {   # frases tipo buscador
        "Q1": ("a detective solving a murder mystery",
               "Novelas de misterio o crimen donde un detective investiga un asesinato."),
        "Q2": ("a princess in a castle with a dragon",
               "Cuentos de hadas o historias infantiles con princesas, reyes y dragones."),
        "Q3": ("the history of the roman empire and its emperors",
               "Libros de historia sobre Roma antigua, sus emperadores y su imperio."),
        "Q4": ("how to cook and bake with flour and sugar",
               "Libros de cocina o recetas para preparar pan, pasteles y otros platos."),
    },
}

In [27]:
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

def vectorizar_consulta(texto, vectorizador, indice_vocab):
    # Preprocesamiento
    tokens = preprocess(texto)
    q_sparse = vectorizador.transform([" ".join(tokens)])
    q = q_sparse.toarray()[0]
    fuera_vocab = [t for t in tokens if t not in indice_vocab]
    return tokens, q_sparse, q, fuera_vocab

def normas_de_documentos(X):
    return np.array([np.linalg.norm(fila) for fila in X])

def similitud_coseno(q, X, normas_docs):
    # sim(q,d) = (q . d) / (||q|| * ||d||)
    productos = X @ q
    norma_q = np.linalg.norm(q)
    if norma_q == 0:
        return np.zeros(X.shape[0])
    return productos / (normas_docs * norma_q)

normas_docs = normas_de_documentos(X_tfidf)
X_sparse = csr_matrix(X_tfidf)               # versión dispersa

In [28]:
TOLERANCIA = 1e-9

def procesar_consulta(nombre, consultas):
    # Vectoriza cada consulta, calcula el coseno propio y lo compara con scikit-learn
    resultados_consulta = {}
    print(f"\n === Consulta: {nombre} ===")
    for id_q, (texto, necesidad) in consultas.items():
        tokens, q_sparse, q, fuera = vectorizar_consulta(texto, vectorizador, indice_vocab)
        sims = similitud_coseno(q, X_tfidf, normas_docs)
        sims_sk = cosine_similarity(q_sparse, X_sparse)[0]
        dif = np.abs(sims - sims_sk).max()

        resultados_consulta[id_q] = {"texto": texto, "necesidad": necesidad,
                                  "tokens": tokens, "q": q, "sims": sims}
        print(f"{id_q}: {texto}")
        print(f"   Necesidad: {necesidad}")
        print(f"   Tokens: {tokens} | fuera del vocabulario: {fuera}")
        print(f"   Diferencia máxima con scikit-learn: {dif:.2e} | coinciden: {dif < TOLERANCIA}")
    return resultados_consulta

resultados = {nombre: procesar_consulta(nombre, grupo) for nombre, grupo in consultas.items()}


 === Consulta: clave ===
Q1: detective murder mystery clue police
   Necesidad: Novelas de misterio o crimen donde un detective investiga un asesinato.
   Tokens: ['detective', 'murder', 'mystery', 'clue', 'police'] | fuera del vocabulario: []
   Diferencia máxima con scikit-learn: 1.60e-16 | coinciden: True
Q2: fairy tale princess king castle dragon
   Necesidad: Cuentos de hadas o historias infantiles con princesas, reyes y dragones.
   Tokens: ['fairy', 'tale', 'princess', 'king', 'castle', 'dragon'] | fuera del vocabulario: []
   Diferencia máxima con scikit-learn: 1.73e-16 | coinciden: True
Q3: ancient rome emperor empire senate
   Necesidad: Libros de historia sobre Roma antigua, sus emperadores y su imperio.
   Tokens: ['ancient', 'rome', 'emperor', 'empire', 'senate'] | fuera del vocabulario: []
   Diferencia máxima con scikit-learn: 3.24e-16 | coinciden: True
Q4: recipe cook bake flour sugar
   Necesidad: Libros de cocina o recetas para preparar pan, pasteles y otros platos.


In [29]:
def detalle_consulta_documento(consulta, id_q, fila_doc):
    # Desglose del coseno: términos compartidos, productos parciales, producto punto, normas y similitud
    r = resultados[consulta][id_q]
    q = r["q"]
    terminos = sorted(set(t for t in r["tokens"] if t in indice_vocab))
    filas = []
    for t in terminos:
        j = indice_vocab[t]
        if X_tfidf[fila_doc, j] > 0:
            filas.append((t, q[j], X_tfidf[fila_doc, j], q[j] * X_tfidf[fila_doc, j]))
    tabla = pd.DataFrame(filas, columns=["Término compartido", "Peso en q", "Peso en d", "Producto parcial"])

    producto = tabla["Producto parcial"].sum()
    norma_q = np.linalg.norm(q)
    norma_d = normas_docs[fila_doc]
    doc_id = ids_docs[fila_doc]

    print(f"[{consulta}] {id_q} vs documento {doc_id}: {titulos[doc_id]}")
    display(tabla)
    print(f"Producto punto : {producto:.4f}")
    print(f"||q||          : {norma_q:.4f}")
    print(f"||d||          : {norma_d:.4f}")
    print(f"Similitud      : {producto / (norma_q * norma_d):.6f}")
    print(f"Verificación   : {r['sims'][fila_doc]:.6f}")

# Ejemplo mejor documento de en cada consulta
for nombre in ["clave"]:
    for id_q, r in resultados[nombre].items():
        fila_top = int(np.argmax(r["sims"]))
        detalle_consulta_documento(nombre, id_q, fila_top)
        print()

[clave] Q1 vs documento 59428: The Florentine Dagger: A Novel for Amateur Detectives


,Término compartido,Peso en q,Peso en d,Producto parcial
0,detective,3.454043,162.340025,560.729438
1,murder,1.953895,123.095382,240.515447
2,mystery,1.863608,50.317419,93.771950
3,police,2.186267,65.588019,143.392943


Producto punto : 1038.4098
||q||          : 5.6473
||d||          : 6470.7028
Similitud      : 0.028417
Verificación   : 0.028417

[clave] Q2 vs documento 30024: The Fisher-Boy Urashima


,Término compartido,Peso en q,Peso en d,Producto parcial
0,dragon,2.815290,16.891740,47.555146
1,fairy,2.379241,2.379241,5.660790
2,king,1.612560,3.225120,5.200701
3,princess,2.391363,4.782726,11.437232


Producto punto : 69.8539
||q||          : 5.4476
||d||          : 198.8082
Similitud      : 0.064498
Verificación   : 0.064498

[clave] Q3 vs documento 11021: A Compilation of the Messages and Papers of the Presidents. Volume 5, part 4: James Buchanan


,Término compartido,Peso en q,Peso en d,Producto parcial
0,ancient,1.588596,19.063151,30.283643
1,emperor,2.355431,21.198877,49.932488
2,empire,2.073648,41.472969,86.000357
3,rome,2.035908,2.035908,4.144922
4,senate,3.109203,1097.548514,3412.500681


Producto punto : 3582.8621
||q||          : 5.1170
||d||          : 18203.3967
Similitud      : 0.038465
Verificación   : 0.038465

[clave] Q4 vs documento 63283: New Mexico Magazine's A Taste of New Mexico Kitchens


,Término compartido,Peso en q,Peso en d,Producto parcial
0,bake,3.785820,49.215666,186.321676
1,cook,1.975116,47.402782,93.625989
2,flour,2.726210,111.774624,304.721135
3,recipe,3.853643,65.511932,252.459603
4,sugar,2.216522,62.062608,137.563119


Producto punto : 974.6915
||q||          : 6.7401
||d||          : 1083.4480
Similitud      : 0.133472
Verificación   : 0.133472



### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [31]:
def fragmento(doc_id, tokens, ancho=100):
    texto = textos[doc_id]
    terminos = {t for t in tokens if idf.get(t, 0) >= 1.5}   # ignora palabras muy comunes
    if not terminos:
        terminos = set(tokens)
    patron = re.compile(r"\b(" + "|".join(map(re.escape, terminos)) + r")\b", re.IGNORECASE)

    mejor, mejor_n = None, 0
    for m in patron.finditer(texto):                          # recorre todas las apariciones
        ini, fin = max(0, m.start() - ancho), min(len(texto), m.end() + ancho)
        n = len(terminos & set(preprocess(texto[ini:fin])))   # términos distintos en la ventana
        if n > mejor_n:
            mejor, mejor_n = (ini, fin), n
    if mejor is None:
        return " ".join(texto[:2 * ancho].split()) + "..."
    return "..." + " ".join(texto[mejor[0]:mejor[1]].split()) + "..."

In [32]:
def ranking_tfidf(nombre, id_q, k=5):
    # Ordena los documentos por similitud coseno y devuelve el top-k como tabla
    r = resultados[nombre][id_q]
    sims = r["sims"]

    # Versión con la biblioteca, con el mismo vector de consulta
    q_sparse = vectorizador.transform([" ".join(r["tokens"])])
    sims_sk = cosine_similarity(q_sparse, X_sparse)[0]

    orden = np.argsort(sims)[::-1][:k]
    orden_sk = np.argsort(sims_sk)[::-1][:k]

    filas = []
    for pos, fila in enumerate(orden, start=1):
        doc_id = ids_docs[fila]
        filas.append((pos, doc_id, titulos[doc_id], round(sims[fila], 6),
                      fragmento(doc_id, r["tokens"])))
    tabla = pd.DataFrame(filas, columns=["Posición", "Identificador", "Título", "Puntuación", "Fragmento"])

    dif = np.abs(sims - sims_sk).max()
    mismo_top = list(orden) == list(orden_sk)
    return tabla, dif, mismo_top

In [33]:
pd.set_option("display.max_colwidth", None)

rankings_tfidf = {}
for id_q, r in resultados["clave"].items():
    tabla, dif, mismo_top = ranking_tfidf("clave", id_q)
    rankings_tfidf[("clave", id_q)] = list(tabla["Identificador"])

    print(f"[clave] {id_q}: {r['texto']}")
    display(tabla)
    print(f"Diferencia máxima explícita vs scikit-learn: {dif:.2e} | "
          f"dentro de la tolerancia: {dif < TOLERANCIA} | mismo top-5: {mismo_top}\n")

[clave] Q1: detective murder mystery clue police


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,59428,The Florentine Dagger: A Novel for Amateur Detectives,0.028417,"...Insurance often isn’t collectable in a case of suicide, so he camouflaged the thing to look like murder.” Again De Medici’s mind played with the words of the detective. The man was either feigning or..."
1,2,61834,The City of Dreadful Night,0.019501,"...ks of the Hugli, 37 CHAPTER V. With the Calcutta Police, 49 CHAPTER VI. The City of Dreadful Night,..."
2,3,434,The Circular Staircase,0.018700,"...secret room, at the very least, and I think even Mr. Jamieson had fancied he might at last have a clue to the mystery. There was evidently nothing more to be discovered: Liddy reported that everything..."
3,4,26176,The Secret House,0.018237,"...he brother knows, but our friend Montague will have it that the world should know. He threatens to murder, if necessary, should she betray his demands to the police. This is not the first time he has utt..."
4,5,78153,Thirteen days,0.016062,"...ich were taboo and under guard. It would also have been impossible not to have enjoyed the Boston police on that day. They were such nice-looking, well-dressed men who but gained in interest by the fact..."


Diferencia máxima explícita vs scikit-learn: 1.60e-16 | dentro de la tolerancia: True | mismo top-5: True

[clave] Q2: fairy tale princess king castle dragon


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,30024,The Fisher-Boy Urashima,0.064498,"...hter took the other; and they rowed, and they rowed, and they rowed till at last they came to the Dragon Palace where the Sea-God lived and ruled as King over all the dragons and the tortoises and the fi..."
1,2,11592,Children's Hour with Red Riding Hood and Other Stories,0.064449,"...the Waiting-woman from her horse, thinking she was his bride. She was led upstairs, but the real Princess had to stay below. The old King looked out of the window and saw the delicate, pretty little cr..."
2,3,49139,The Life of Carmen Sylva (Queen of Roumania),0.056064,"...ke of Ostgothland, the present reigning King of Sweden. Tutors and governesses had now left the Castle. Pastor Harder, a clergyman from Neuwied, came daily to Princess Elizabeth to lecture upon logic,..."
3,4,21635,Prudence Says So,0.041120,"...t. And with a cry of relief, and shame, and fear, the twins plunged upon her and told their little tale. ""You punish us this time, Fairy,"" begged Carol. ""We--we don't want the rest of the family to k..."
4,5,64374,Twenty-Seven Drawings by William Blake,0.036017,"...We are indebted to the Fogg Art Museum for the privilege of reproducing “The Angel Binding the Dragon”; to the Metropolitan Museum of Art for “She Shall be Called Woman”, “The Flight into Egypt”, “The..."


Diferencia máxima explícita vs scikit-learn: 1.73e-16 | dentro de la tolerancia: True | mismo top-5: True

[clave] Q3: ancient rome emperor empire senate


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,11021,"A Compilation of the Messages and Papers of the Presidents. Volume 5, part 4: James Buchanan",0.038465,"...e, the letter of credence from the President was received with peculiar honors by Kweiliang, ""the Emperor's prime minister and the second man in the Empire to the Emperor himself."" The ratifications of th..."
1,2,6839,The Old Roman World : the Grandeur and Failure of Its Civilization.,0.037437,...itectural Monuments--Roads--Commerce--Agriculture--Manufactures-- Wealth--Population--Unity of the Empire CHAPTER III. THE WONDERS OF ANCIENT ROME. Original Settlement--The Seven Hills--Progress o...
2,3,3801,Napoleon and Blücher: An Historical Novel,0.034386,"...his own tremendous columns in one or two decisive battles, and then, on the ruins of the Russian empire, dictate terms of peace to the humiliated emperor. This has been the course of events ever since..."
3,4,35461,A Short History of the World,0.034182,"...e Bell 172 The Dying Gaul 175 Ancient Roman Cisterns at Carthage 177 Hannibal 181 Roman Empire and its Alliances, 150 B.C. _Map_ 183 The Forum, Rome 188 Ruined Coliseum in Tunis 189 Ro..."
4,5,20298,"History of the Wars, Books V and VI: The Gothic War",0.032381,"...DATE: [F] 526 A.D. Symmachus and his son-in-law Boetius were men of noble and ancient lineage, and both had been leading men[9] in the Roman senate and had been consuls. But because t..."


Diferencia máxima explícita vs scikit-learn: 3.24e-16 | dentro de la tolerancia: True | mismo top-5: True

[clave] Q4: recipe cook bake flour sugar


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,63283,New Mexico Magazine's A Taste of New Mexico Kitchens,0.133472,...s this marvelous cookie recipe using New Mexico’s favorite nuts. 4 eggs 1½ cups granulated sugar ½ teaspoon grated lemon rind 2½ cups sifted flour ¼ teaspoon salt ¼ cup confectioners’...
1,2,6978,Made-Over Dishes,0.088381,"...uble boiler one quart of milk; allow it to cook for thirty minutes; then add two tablespoonfuls of sugar, a grating of nutmeg, and one cup of cold boiled rice; turn this into a baking pan, and bake in a..."
2,3,34822,The Century Cook Book,0.068900,"...them to the water in which they were boiled, and mix them well together. Add two tablespoonfuls of flour, one half cupful of granulated sugar, and one tablespoonful of salt. Cook it for a few minutes, a..."
3,4,54568,A handbook of invalid cooking,0.046958,"...salt. 1 Teaspoon of sugar. 1 Quart of boiling water. 1 Cup of milk. Mix the corn-meal, flour, salt, and sugar into a thin paste with cold water, and pour into it the boiling water. Cook it in..."
4,5,31605,The Cooking Manual of Practical Directions for Economical Every-Day Cookery,0.043939,"...n the gravy made as above into the jar; put on the cover; fasten it in place with a paste made of flour and water, and oiled on the top to prevent cracking. Bake the hare in a moderate oven three hours...."


Diferencia máxima explícita vs scikit-learn: 1.45e-16 | dentro de la tolerancia: True | mismo top-5: True

